# Task 1.1 — Desempeño teórico (M/M/c y M/M/c/c)
Hospital Regional del Altiplano · UCI

In [1]:
# --- Parámetros del grupo ---
import math
g = 3                                   # <-- número de grupo (cambiar aquí)
mu = 0.25
lam = 3.0 + 0.2 * ((g - 1) % 5)
c = 17 + ((g - 1) % 3)
a = lam / mu
rho = a / c
k_off = math.ceil(c - a) + 3
Tb = 7
print(f"Grupo {g}: lambda={lam:.1f}/día, c={c}, a={a:.2f}, rho={rho:.4f}, k_off={k_off}, Tb={Tb} días")

Grupo 3: lambda=3.4/día, c=19, a=13.60, rho=0.7158, k_off=9, Tb=7 días


## Fórmulas
**Con espera (M/M/c)**, $a=\lambda/\mu$, $\rho=a/c<1$:

$$B(c,a):\; B_0=1,\; B_k=\frac{aB_{k-1}}{k+aB_{k-1}},\qquad C(c,a)=\frac{B(c,a)}{1-\rho\,(1-B(c,a))}=\frac{\frac{a^c}{c!}\frac{1}{1-\rho}}{\sum_{k=0}^{c-1}\frac{a^k}{k!}+\frac{a^c}{c!}\frac{1}{1-\rho}}$$

$$L_q=C\,\frac{\rho}{1-\rho},\quad W_q=\frac{L_q}{\lambda}=\frac{C}{c\mu-\lambda},\quad W=W_q+\frac1\mu,\quad L=L_q+a$$

**Sin espera (M/M/c/c):** bloqueo $B(c,a)$ (recurrencia), $\lambda_{ef}=\lambda(1-B)$, camas ocupadas $L=\lambda_{ef}/\mu=a(1-B)$, utilización $L/c$.

In [2]:
import numpy as np

def erlang_b(c, a):
    """Recurrencia: B(0)=1;  B(k) = a B(k-1) / (k + a B(k-1))"""
    B = 1.0
    for k in range(1, c + 1):
        B = a * B / (k + a * B)
    return B

def erlang_c_direct(c, a):
    """C(c,a) = [a^c/c! /(1-rho)] / [sum_{k<c} a^k/k! + a^c/c! /(1-rho)]  (estable vía logs)"""
    from math import lgamma, log, exp
    rho = a / c
    terms = [exp(k*log(a) - lgamma(k+1)) for k in range(c)]
    top = exp(c*log(a) - lgamma(c+1)) / (1 - rho)
    return top / (sum(terms) + top)

def theory_wait(lam, mu, c):
    a = lam / mu; rho = a / c
    assert rho < 1
    B = erlang_b(c, a)
    C = B / (1 - rho * (1 - B))             # Erlang C a partir de Erlang B
    Lq = C * rho / (1 - rho)
    Wq = Lq / lam                            # = C/(c mu - lam)
    W = Wq + 1 / mu
    L = Lq + a
    return dict(a=a, rho=rho, C=C, Lq=Lq, Wq_h=Wq*24, W_h=W*24, Wq=Wq, W=W, L=L, util=rho, Pwait=C)

def theory_loss(lam, mu, c):
    a = lam / mu
    B = erlang_b(c, a)
    lam_eff = lam * (1 - B)
    L = lam_eff / mu                         # = a (1-B)
    return dict(B=B, lam_eff=lam_eff, L=L, util=L / c, W=1/mu, Pblock=B)


In [3]:
import pandas as pd
tw, tl = theory_wait(lam, mu, c), theory_loss(lam, mu, c)
print("Erlang C directo vs. vía Erlang B:", erlang_c_direct(c, a), tw['C'])
df_w = pd.DataFrame({
    "carga a": tw['a'], "rho": tw['rho'], "C(c,a)=P(esperar)": tw['C'], "Lq": tw['Lq'],
    "Wq [h]": tw['Wq_h'], "W [h]": tw['W_h'], "L": tw['L']}, index=["M/M/c"]).T
df_w

Erlang C directo vs. vía Erlang B: 0.12035283678624603 0.12035283678624643


,M/M/c
carga a,13.600000
rho,0.715789
"C(c,a)=P(esperar)",0.120353
Lq,0.303111
Wq [h],2.139606
W [h],98.139606
L,13.903111


In [4]:
df_l = pd.DataFrame({"B(c,a)": tl['B'], "lambda_ef [pac/día]": tl['lam_eff'],
                     "camas ocupadas L": tl['L'], "utilización L/c": tl['util']}, index=["M/M/c/c"]).T
df_l

,M/M/c/c
"B(c,a)",0.037430
lambda_ef [pac/día],3.272738
camas ocupadas L,13.090951
utilización L/c,0.688997


## Ley de Little ($L=\lambda_{ef}\,W$, $L_q=\lambda\,W_q$)

In [5]:
print("M/M/c   : L =", tw['L'], " lam*W =", lam*tw['W'], " | Lq =", tw['Lq'], " lam*Wq =", lam*tw['Wq'])
print("M/M/c/c : L =", tl['L'], " lam_ef*(1/mu) =", tl['lam_eff']/mu)
print("Error si se usa lambda en vez de lambda_ef (M/M/c/c): lam/mu =", lam/mu, "vs L =", tl['L'],
      f"-> sobreestima {100*(lam/mu/tl['L']-1):.2f}% (= B/(1-B))")
assert abs(tw['L'] - lam*tw['W']) < 1e-9 and abs(tl['L'] - tl['lam_eff']/mu) < 1e-9

M/M/c   : L = 13.903110848202399  lam*W = 13.903110848202397  | Lq = 0.3031108482023984  lam*Wq = 0.3031108482023984
M/M/c/c : L = 13.09095145993353  lam_ef*(1/mu) = 13.09095145993353
Error si se usa lambda en vez de lambda_ef (M/M/c/c): lam/mu = 13.6 vs L = 13.09095145993353 -> sobreestima 3.89% (= B/(1-B))


**Una línea:** usar $\lambda$ en vez de $\lambda_{ef}$ en la variante sin espera sobreestima camas ocupadas y utilización por el factor $1/(1-B)$ (cuenta como atendidos a los pacientes desviados).